# 01 · Prepare RDD2022 (India subset)

> **OWNER:** Member A (M1 · road defects)
> **PREREQUISITES:** `00_setup_verify.ipynb` all-green.
> **EXPECTED RUNTIME:** 1-3 hours, almost entirely the download+extract — the
> conversion/split/contact-sheet steps run in a few minutes once the raw data
> is on disk.
> **OUTPUTS:** `data/rdd2022_india/{images,labels}/{train,val,test}/` + `data.yaml`,
> a contact sheet PNG, and printed class-balance / on-disk-size reports.

Source: figshare RDD2022, DOI [10.6084/m9.figshare.21431547](https://doi.org/10.6084/m9.figshare.21431547),
and [github.com/sekilab/RoadDamageDetector](https://github.com/sekilab/RoadDamageDetector)
(loader scripts + the same data).

**Next notebook:** `02_train_road_damage.ipynb` (yours too).

In [ ]:
!git clone https://github.com/ad8thya/SmartIndiaHackathon.git

In [ ]:
# Cell 1/3 — minimal bootstrap (Colab vs local). No repo imports yet: on a
# fresh Colab runtime nothing has been cloned, so this cell is deliberately
# self-contained and only prepares sys.path so `common/` becomes importable.
import subprocess
import sys
from pathlib import Path


def _in_colab() -> bool:
    try:
        import google.colab  # noqa: F401

        return True
    except ImportError:
        return False


IN_COLAB = _in_colab()
REPO_URL = "https://github.com/ad8thya/SmartIndiaHackathon.git"

if IN_COLAB:
    REPO_ROOT = Path("/content/SmartIndiaHackathon")
    if not REPO_ROOT.exists():
        print(f"cloning {REPO_URL} -> {REPO_ROOT}")
        subprocess.run(["git", "clone", REPO_URL, str(REPO_ROOT)], check=True)
    else:
        print(f"{REPO_ROOT} already present locally on this runtime")
else:
    _here = Path.cwd().resolve()
    _candidates = [c for c in (_here, *_here.parents) if (c / "pyproject.toml").exists() and (c / "notebooks").exists()]
    if not _candidates:
        raise RuntimeError(
            "Could not find the repo root (looked for pyproject.toml + notebooks/ "
            f"walking up from {_here}). Run this notebook from inside the repo checkout."
        )
    REPO_ROOT = _candidates[0]

for _p in (str(REPO_ROOT), str(REPO_ROOT / "notebooks")):
    if _p not in sys.path:
        sys.path.insert(0, _p)

print(f"Colab: {IN_COLAB}")
print(f"repo root: {REPO_ROOT}")

In [ ]:
# Cell 2/3 — install the ML extras. Quiet; ~60-90s on a fresh Colab runtime,
# near-instant if already installed (pip no-ops on a satisfied requirement).
import subprocess
import sys

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-e", f"{REPO_ROOT}[ml]"],
    check=True,
)
print("ml extras installed")

In [ ]:
# Cell 3/3 — full environment setup (Drive mount + DATA_ROOT/MODEL_ROOT) and
# an import check: torch version + CUDA availability, so a broken environment
# fails here, not forty minutes into a training run.
from common import colab as colab_mod

env = colab_mod.setup_environment()
REPO_ROOT, DATA_ROOT, MODEL_ROOT = env["repo_root"], env["data_root"], env["model_root"]

import ultralytics

print(f"ultralytics {ultralytics.__version__}")
gpu_info = colab_mod.gpu_report()

## Step 1 — Get the archive on disk (manual — this notebook does not scrape or auto-download)

The full RDD2022 archive spans six countries (India, Japan, Norway, Czechia,
USA, China) and is many GB. **Look for an India-only per-country file on
figshare first** — it is dramatically smaller and needs no cleanup step.

1. Open the DOI or GitHub link above and find India's data.
2. If a **per-country India-only** archive exists, download it and save it as
   one of the `INDIA_ONLY_CANDIDATES` filenames below, under `RAW_DIR`.
3. **Only if no per-country file exists**, download the full six-country
   archive and save it as one of `FULL_ARCHIVE_CANDIDATES` under `RAW_DIR` —
   the next cell will extract India/ only and **delete the rest immediately**.
   Do not leave Japanese/Norwegian/other-country images on disk.
4. Re-run the next cell once the file is in place.

In [ ]:
RDD_DOI_URL = "https://doi.org/10.6084/m9.figshare.21431547"
RDD_GITHUB = "https://github.com/sekilab/RoadDamageDetector"

RAW_DIR = DATA_ROOT / "raw" / "rdd2022"
RAW_DIR.mkdir(parents=True, exist_ok=True)

INDIA_ONLY_CANDIDATES = ["RDD2022_India.zip", "India.zip", "rdd2022_india.zip"]
FULL_ARCHIVE_CANDIDATES = ["RDD2022_all_countries.zip", "RDD2022.zip", "RDD2022_all.zip"]


def _find_existing(dirpath, names):
    for name in names:
        p = dirpath / name
        if p.exists():
            return p
    for p in dirpath.glob("*.zip"):
        if "india" in p.name.lower():
            return p
    return None


india_zip = _find_existing(RAW_DIR, INDIA_ONLY_CANDIDATES)
full_zip = _find_existing(RAW_DIR, FULL_ARCHIVE_CANDIDATES) if india_zip is None else None

if india_zip is None and full_zip is None:
    raise FileNotFoundError(
        f"No RDD2022 archive found in {RAW_DIR}.\n\n"
        f"  1. Open {RDD_DOI_URL} (figshare) or {RDD_GITHUB} (GitHub mirror + loaders).\n"
        f"  2. PREFER a per-country India archive if one exists — save as one of\n"
        f"     {INDIA_ONLY_CANDIDATES} under {RAW_DIR}.\n"
        f"  3. Otherwise download the FULL six-country archive and save as one of\n"
        f"     {FULL_ARCHIVE_CANDIDATES} under {RAW_DIR} — this notebook extracts\n"
        f"     India/ only and deletes the rest.\n"
        f"  4. Re-run this cell."
    )

## Step 1b — confirm before extracting

Prints the download size and requires an explicit `YES` before touching anything — the full archive is large enough that extracting the wrong thing by accident is expensive to redo.

In [ ]:
import os

target = india_zip or full_zip
size_gb = os.path.getsize(target) / (1024**3)
print(f"found: {target.name}  ({size_gb:.2f} GB)")
if full_zip is not None:
    print("This is the FULL six-country archive.")
    print("We will extract India/ only and DELETE the archive afterward to avoid")
    print("leaving Japanese/Norwegian/Czech/USA/Chinese road images on disk.")

_confirm = input(f"Type YES to proceed with extracting {target.name}: ").strip()
if _confirm != "YES":
    raise SystemExit("not confirmed — stopping. Re-run this cell when ready.")

In [ ]:
import zipfile

EXTRACT_ROOT = RAW_DIR / "extracted"
EXTRACT_ROOT.mkdir(exist_ok=True)

with zipfile.ZipFile(target) as zf:
    if full_zip is not None:
        india_members = [m for m in zf.namelist() if "india" in m.lower()]
        if not india_members:
            raise RuntimeError(
                "Full archive found but no member path contains 'India' — "
                "inspect zf.namelist() manually, the naming may differ from what this notebook expects."
            )
        print(f"extracting {len(india_members)} India-only members...")
        for m in india_members:
            zf.extract(m, EXTRACT_ROOT)
    else:
        print(f"extracting {target.name} (already India-only)...")
        zf.extractall(EXTRACT_ROOT)

if full_zip is not None:
    print(f"deleting full six-country archive {full_zip} — India/ has been extracted, the rest is not needed")
    full_zip.unlink()

print(f"extraction complete: {EXTRACT_ROOT}")

## Step 1c — locate the images/ and annotations/ folders

RDD2022's India subset ships as `India/train/images/*.jpg` +
`India/train/annotations/xmls/*.xml` (folder naming has drifted slightly
between RDD2022 releases). This cell searches for them; **if it can't find
them automatically, set `IMAGES_DIR`/`ANNOTATIONS_DIR` by hand** after
inspecting `EXTRACT_ROOT` yourself.

In [ ]:
def _find_dir(root, name_contains):
    candidates = [p for p in root.rglob("*") if p.is_dir() and name_contains in p.name.lower()]
    return candidates[0] if candidates else None


IMAGES_DIR = _find_dir(EXTRACT_ROOT, "image")
ANNOTATIONS_DIR = _find_dir(EXTRACT_ROOT, "xml") or _find_dir(EXTRACT_ROOT, "annotation")

print(f"IMAGES_DIR:      {IMAGES_DIR}")
print(f"ANNOTATIONS_DIR: {ANNOTATIONS_DIR}")

if IMAGES_DIR is None or ANNOTATIONS_DIR is None:
    print()
    print("Could not auto-locate both directories. List EXTRACT_ROOT and set them by hand:")
    print(f"  list(EXTRACT_ROOT.rglob('*'))[:40]  # EXTRACT_ROOT = {EXTRACT_ROOT}")

## Step 2 — Inspect before converting

Image/XML pairing, resolution distribution, and — **printed prominently** —
instances per class. RDD2022's D40 (POTHOLE) is the minority class, and
POTHOLE is the class the entire pitch is about. If the ratio below is severe,
notebook 02's second training run addresses it explicitly — but you need the
real number first.

In [ ]:
import xml.etree.ElementTree as ET
from collections import Counter

from PIL import Image

_image_stems = {p.stem for p in IMAGES_DIR.iterdir() if p.suffix.lower() in (".jpg", ".jpeg", ".png")}
_xml_stems = {p.stem for p in ANNOTATIONS_DIR.glob("*.xml")}

orphan_images = sorted(_image_stems - _xml_stems)
orphan_xml = sorted(_xml_stems - _image_stems)
print(f"images: {len(_image_stems)}   xml: {len(_xml_stems)}")
print(f"orphan images (no xml): {len(orphan_images)}  e.g. {orphan_images[:5]}")
print(f"orphan xml (no image):  {len(orphan_xml)}  e.g. {orphan_xml[:5]}")

resolutions = Counter()
class_counts = Counter()
_paired = sorted(_image_stems & _xml_stems)
for stem in _paired:
    img_path = next(p for p in IMAGES_DIR.iterdir() if p.stem == stem)
    with Image.open(img_path) as img:
        resolutions[img.size] += 1
    root = ET.parse(ANNOTATIONS_DIR / f"{stem}.xml").getroot()
    for obj in root.findall("object"):
        name_el = obj.find("name")
        if name_el is not None and name_el.text:
            class_counts[name_el.text.strip()] += 1

print()
print("resolution distribution (top 5):")
for res, n in resolutions.most_common(5):
    print(f"  {res}: {n} images")

print()
print("=" * 60)
print("CLASS BALANCE (all D-codes present in the raw XML, before filtering to the frozen 4):")
total = sum(class_counts.values()) or 1
for name, n in class_counts.most_common():
    marker = "  <-- frozen class" if name in ("D00", "D10", "D20", "D40") else ""
    print(f"  {name:12s} {n:6d}  ({100 * n / total:5.1f}%){marker}")
print("=" * 60)
d40 = class_counts.get("D40", 0)
d00 = class_counts.get("D00", 0)
if d40 and d00:
    print(f"D40 (POTHOLE) is {d40 / max(d00, 1):.2f}x the count of D00 (LONGITUDINAL_CRACK) — "
          f"{'a severe imbalance' if d40 / max(d00, 1) < 0.3 else 'imbalanced, note it'}. "
          "Notebook 02 trains a second run addressing this directly.")

## Step 3 — Convert VOC -> YOLO, frozen indices

Only the four frozen classes (D00/D10/D20/D40) are kept — every other D-code
present in the raw data is reported as `unknown_classes` and dropped, not
silently folded into a class it doesn't belong to.

In [ ]:
from common import constants, voc_to_yolo

CLASS_MAP = {code_name: idx for idx, code_name in constants.RDD_CLASSES.items()}  # {"D00": 0, "D10": 1, "D20": 2, "D40": 3}
print(f"class_map (VOC name -> frozen index): {CLASS_MAP}")

CONVERTED_LABELS_DIR = DATA_ROOT / "raw" / "rdd2022" / "labels_yolo"
report = voc_to_yolo.convert_voc_dir(
    images_dir=IMAGES_DIR,
    annotations_dir=ANNOTATIONS_DIR,
    class_map=CLASS_MAP,
    output_labels_dir=CONVERTED_LABELS_DIR,
)
report.print_summary()

## Step 4 — Stratified 80/10/10 split (seed=42)

Stratified by each image's rarest present class so D40 (POTHOLE) doesn't concentrate in one split.

In [ ]:
from common import splits

image_class_map = {}
for stem in _paired:
    label_path = CONVERTED_LABELS_DIR / f"{stem}.txt"
    classes = set()
    if label_path.exists():
        for line in label_path.read_text().splitlines():
            if line.strip():
                classes.add(int(line.split()[0]))
    image_class_map[stem] = classes

data_splits = splits.stratified_split(image_class_map, ratios=(0.8, 0.1, 0.1), seed=42)
splits.report_split_balance(image_class_map, data_splits, constants.RDD_DETECTION_NAMES)

## Step 5 — Materialize the split + write data.yaml

In [ ]:
import yaml

OUTPUT_ROOT = DATA_ROOT / "rdd2022_india"
splits.materialize_split(
    image_class_map=image_class_map,
    splits=data_splits,
    images_src=IMAGES_DIR,
    labels_src=CONVERTED_LABELS_DIR,
    output_root=OUTPUT_ROOT,
)

data_yaml = {
    "path": str(OUTPUT_ROOT),
    "train": "images/train",
    "val": "images/val",
    "test": "images/test",
    "names": constants.RDD_DETECTION_NAMES,
}
constants.assert_class_order(data_yaml["names"], constants.RDD_DETECTION_NAMES, "rdd")

data_yaml_path = OUTPUT_ROOT / "data.yaml"
data_yaml_path.write_text(yaml.safe_dump(data_yaml, sort_keys=False))
print(f"wrote {data_yaml_path}")
print(data_yaml_path.read_text())

## Step 6 — Contact sheet — LOOK AT THIS BEFORE ANYTHING ELSE

12 random training images with boxes drawn. **Offset or inverted boxes mean
the VOC->YOLO conversion is broken** — training will run happily on broken
boxes and produce a model with a normal-looking loss curve that is useless in
the field. Ten seconds here saves a day of debugging why the trained model is
garbage.

In [ ]:
from common import contact_sheet

sheet_path = contact_sheet.render_contact_sheet(
    images_dir=OUTPUT_ROOT / "images" / "train",
    labels_dir=OUTPUT_ROOT / "labels" / "train",
    class_names=constants.RDD_DETECTION_NAMES,
    output_path=OUTPUT_ROOT / "contact_sheet.png",
    n=12,
)

from IPython.display import Image as IPImage, display

display(IPImage(filename=str(sheet_path)))

## Step 7 — gitignore check + on-disk size

In [ ]:
def _dir_size_gb(path):
    return sum(p.stat().st_size for p in Path(path).rglob("*") if p.is_file()) / (1024**3)


dataset_size_gb = _dir_size_gb(OUTPUT_ROOT)
raw_size_gb = _dir_size_gb(RAW_DIR)
print(f"prepared dataset ({OUTPUT_ROOT}): {dataset_size_gb:.2f} GB")
print(f"raw download+extract ({RAW_DIR}):  {raw_size_gb:.2f} GB")

gitignore_path = REPO_ROOT / ".gitignore"
gitignore_text = gitignore_path.read_text()
if "data/rdd2022_india/" in gitignore_text:
    print(".gitignore already excludes data/rdd2022_india/ — nothing to do")
else:
    print("WARNING: data/rdd2022_india/ is not in .gitignore — add it before `git add`ing anything")

---
### What this notebook produced
- `data/rdd2022_india/{images,labels}/{train,val,test}/` + `data.yaml` (gitignored)
- A contact sheet PNG you looked at and confirmed looks correct
- The real D40/D00 class-balance ratio, printed prominently

### Next
`02_train_road_damage.ipynb` (yours too — M1).